
Basic prompt caching example with the OpenAI API.

What this shows:
  1. A large, static system prompt (>= 1024 tokens) placed at the START of the prompt.
  2. Two calls to the model with the SAME static prefix but DIFFERENT user questions.
  3. OpenAI's caching is AUTOMATIC — no special parameter needed. The system detects
     the repeated prefix and reuses it.
  4. We print usage metrics so you can see `cached_tokens` jump on the second call.

How OpenAI caching works (key facts):
  - Triggers automatically once the prompt is >= 1024 tokens.
  - Cache hits happen in 128-token increments.
  - Static content MUST come first; dynamic (user query) goes last.
  - TTL is roughly 5-10 minutes of inactivity.
  - Cached input tokens are billed at ~50% of normal input price.
  - No code changes are needed beyond structuring the prompt correctly.

Setup:
  1. pip install openai python-dotenv
  2. Create a file named `.env` in the same directory with this line:
        OPENAI_API_KEY=sk-proj-...your-key-here...
  3. python openai_prompt_caching.py


In [1]:
import os
import time
from dotenv import load_dotenv
from openai import OpenAI

# Load variables from .env into os.environ
load_dotenv()

# The OpenAI client automatically reads OPENAI_API_KEY from the environment,
# but we pass it explicitly here to make the flow obvious.
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError(
        "OPENAI_API_KEY not found. Create a .env file with: OPENAI_API_KEY=sk-..."
    )

client = OpenAI(api_key=api_key)

In [2]:
# Static prefix — must be >= 1024 tokens for caching to kick in.
# In a real app this would be your system instructions, tool docs, RAG context, etc.
LARGE_SYSTEM_PROMPT = """
You are a customer support assistant for ACME Robotics.

Product catalog:
- ACME-100: entry-level cleaning robot, $299. Battery life 90 minutes,
  works on hard floors and low-pile carpet. 1-year warranty.
- ACME-200: mid-tier robot with mopping, $499. Battery life 120 minutes,
  HEPA filter, app control, scheduling. 2-year warranty.
- ACME-300: flagship robot with AI navigation, $899. LiDAR mapping,
  obstacle avoidance, self-emptying base, 180-minute battery. 3-year warranty.

Return policy: 30-day money-back guarantee on all unopened products.
Opened products may be returned within 14 days for a 15% restocking fee.
Shipping is free on orders over $99 within the continental US.

Support hours: Monday-Friday 8am-8pm ET, Saturday 9am-5pm ET.
Always greet the customer politely, confirm their issue, and offer next steps.
""" * 30  # repeat to comfortably exceed the 1024-token threshold

In [3]:
def ask(question: str, label: str):
    """Send one chat completion and print cache-related usage metrics."""
    start = time.perf_counter()

    response = client.chat.completions.create(
        model="gpt-4o-mini",  # any chat model that supports caching works
        messages=[
            # Static content FIRST — this is what gets cached.
            {"role": "system", "content": LARGE_SYSTEM_PROMPT},
            # Dynamic content LAST — this is what differs per request.
            {"role": "user", "content": question},
        ],
        max_tokens=200,
    )

    elapsed = time.perf_counter() - start
    usage = response.usage

    # cached_tokens lives inside prompt_tokens_details on modern SDK versions.
    cached = 0
    details = getattr(usage, "prompt_tokens_details", None)
    if details is not None:
        cached = getattr(details, "cached_tokens", 0) or 0

    print(f"\n--- {label} ---")
    print(f"Question        : {question}")
    print(f"Elapsed         : {elapsed:.2f}s")
    print(f"prompt_tokens   : {usage.prompt_tokens}")
    print(f"  cached_tokens : {cached}   <-- this is the cache hit count")
    print(f"completion_tokens: {usage.completion_tokens}")
    print(f"Answer          : {response.choices[0].message.content[:160]}...")

In [4]:
if __name__ == "__main__":
    # FIRST CALL: cache gets written. Expect cached_tokens = 0.
    ask("What's your entry level robot?", label="First call (cache MISS)")

    # SECOND CALL within ~5 min: same prefix, new question.
    # Expect cached_tokens to be a large fraction of prompt_tokens,
    # and the request to feel noticeably snappier.
    ask("Does the ACME-300 have LiDAR?", label="Second call (cache HIT)")

    # THIRD CALL: another new question, prefix still cached.
    ask("What's the return policy on opened items?", label="Third call (cache HIT)")


--- First call (cache MISS) ---
Question        : What's your entry level robot?
Elapsed         : 7.88s
prompt_tokens   : 6228
  cached_tokens : 0   <-- this is the cache hit count
completion_tokens: 68
Answer          : Hello! Our entry-level robot is the **ACME-100**. It is priced at $299 and has a battery life of 90 minutes. It works well on hard floors and low-pile carpet, a...

--- Second call (cache HIT) ---
Question        : Does the ACME-300 have LiDAR?
Elapsed         : 2.38s
prompt_tokens   : 6232
  cached_tokens : 6144   <-- this is the cache hit count
completion_tokens: 45
Answer          : Yes, the ACME-300 has LiDAR mapping, which allows for advanced navigation and obstacle avoidance. If you have any more questions about the ACME-300 or need assi...

--- Third call (cache HIT) ---
Question        : What's the return policy on opened items?
Elapsed         : 2.81s
prompt_tokens   : 6230
  cached_tokens : 6144   <-- this is the cache hit count
completion_tokens: 73
Answer